In [9]:
import pandas as pd
import numpy as np
from ai_edge_litert.interpreter import Interpreter
from collections import Counter

# ==========================================
# 1. LOAD DATA (12 CHANNELS - RAW)
# ==========================================
path = "../../data/hodanje/" # Recording from Sensor Logger Android app

# Load all required sensors
df_ori = pd.read_csv(path + 'Orientation.csv').sort_values('time')
df_grav = pd.read_csv(path + 'Gravity.csv').sort_values('time')
df_gyro = pd.read_csv(path + 'Gyroscope.csv').sort_values('time')

# Bypass the smoothed Accelerometer.csv and load the raw total acceleration
df_tot_acc = pd.read_csv(path + 'TotalAcceleration.csv').sort_values('time')

# Merge all files (asof merge)
df = pd.merge_asof(df_ori[['time', 'roll', 'pitch', 'yaw']], df_grav[['time', 'x', 'y', 'z']], on='time', suffixes=('', '_grav'))
df = pd.merge_asof(df, df_gyro[['time', 'x', 'y', 'z']], on='time', suffixes=('', '_gyro'))
df = pd.merge_asof(df, df_tot_acc[['time', 'x', 'y', 'z']], on='time', suffixes=('', '_tot_acc'))

# Map column names
df.columns = ['time', 'attitude.roll', 'attitude.pitch', 'attitude.yaw',
              'raw_gravity.x', 'raw_gravity.y', 'raw_gravity.z',
              'rotationRate.x', 'rotationRate.y', 'rotationRate.z',
              'raw_total_acc.x', 'raw_total_acc.y', 'raw_total_acc.z']

# ==========================================
# 2. RESAMPLING TO EXACTLY 50 Hz
# ==========================================
df['time_dt'] = pd.to_datetime(df['time'])
df = df.set_index('time_dt')
df = df.resample('20ms').mean().interpolate(method='linear')
df = df.reset_index(drop=True)

# ==========================================
# 3. CONVERSION Android -> iOS CoreMotion
# ==========================================
# Reference:
#   Sensor Logger COORDINATES.md and CROSSPLATFORM.md
#   (https://github.com/tszheichoi/awesome-sensor-logger)
#   Apple CMDeviceMotion docs (gravity face-up = (0,0,-1) g)
# Summary:
#   - Accelerations (gravity, total/linear): m/s² → g, all three axes flip sign
#   - Gyroscope (rotationRate): identity (rad/s, same sign convention)
#   - Attitude roll:  identity (same convention)
#   - Attitude pitch: sign flip (Android CW positive, iOS CCW positive)
#   - Attitude yaw:   sign flip + relativization (Android magnetic, iOS true north)
g = 9.80665

# A) Linear acceleration: total - gravity (both in m/s²)
df['raw_linear_acc.x'] = df['raw_total_acc.x'] - df['raw_gravity.x']
df['raw_linear_acc.y'] = df['raw_total_acc.y'] - df['raw_gravity.y']
df['raw_linear_acc.z'] = df['raw_total_acc.z'] - df['raw_gravity.z']

# B) Accelerations → iOS g with a single per-axis flip
df['gravity.x'] = -df['raw_gravity.x'] / g
df['gravity.y'] = -df['raw_gravity.y'] / g
df['gravity.z'] = -df['raw_gravity.z'] / g

df['userAcceleration.x'] = -df['raw_linear_acc.x'] / g
df['userAcceleration.y'] = -df['raw_linear_acc.y'] / g
df['userAcceleration.z'] = -df['raw_linear_acc.z'] / g

# C) Pitch + yaw flip; yaw then relativized to avoid the
# Android-magnetic-north vs. iOS-true-north problem
df['attitude.pitch'] = -df['attitude.pitch']
df['attitude.yaw']   = -df['attitude.yaw']
df['attitude.yaw']   = df['attitude.yaw'] - df['attitude.yaw'].iloc[0]

# D) Keep only the 12 channels the network expects, in the exact order
df_final = df[['time', 
               'attitude.roll', 'attitude.pitch', 'attitude.yaw',
               'gravity.x', 'gravity.y', 'gravity.z', 
               'rotationRate.x', 'rotationRate.y', 'rotationRate.z', 
               'userAcceleration.x', 'userAcceleration.y', 'userAcceleration.z']]

# ==========================================
# 4. SLIDING WINDOW
# ==========================================
df_clean = df_final.iloc[150:-150].reset_index(drop=True) # trim the first and last 3 seconds
data = df_clean.drop(columns=['time']).values  # (N, 12)

window_size = 128
step_size = 64  # 50% overlap
predictions = []

interpreter = Interpreter(model_path='../../models/cnn_best.tflite')
interpreter.allocate_tensors()
input_idx = interpreter.get_input_details()[0]['index']
output_idx = interpreter.get_output_details()[0]['index']
expected_shape = interpreter.get_input_details()[0]['shape']

# Safety check
assert list(expected_shape) == [1, 128, 12], f"Model expects shape {expected_shape}, but we have 12 channels!"

labels = ["Downstairs", "Upstairs", "Walking", "Jogging", "Standing", "Sitting"]

for start in range(0, len(data) - window_size + 1, step_size):
    window = data[start:start+window_size].reshape(1, 128, 12)
    
    eps = 1e-8
    window_static = window[:, :, 0:6]  # 6 static channels (attitude + gravity)
    window_dynamic = window[:, :, 6:12] # 6 dynamic channels (gyro + acc)
    
    mean = window_dynamic.mean(axis=1, keepdims=True)
    std = window_dynamic.std(axis=1, keepdims=True)
    window_dynamic_n = (window_dynamic - mean) / (std + eps)
    
    X_test_wild = np.concatenate([window_static, window_dynamic_n], axis=2).astype(np.float32)
    
    interpreter.set_tensor(input_idx, X_test_wild)
    interpreter.invoke()
    pred = interpreter.get_tensor(output_idx)
    predicted_class = pred.argmax(axis=1)[0]
    predictions.append((labels[predicted_class], pred[0].copy()))

# ==========================================
# 5. RESULTS AND MAJORITY VOTE
# ==========================================
print(f"--- STATISTICS FOR THE RECORDING ({path}) ---")
labels_only = [p[0] for p in predictions]
counts = Counter(labels_only)
total_windows = len(predictions)

for label, count in counts.most_common():
    percentage = (count / total_windows) * 100
    print(f"{label}: {count}/{total_windows} windows ({percentage:.1f}%)")

print("\n==============================================")
print(f"FINAL APP DECISION: {counts.most_common(1)[0][0]}")
print("==============================================")


--- STATISTICS FOR THE RECORDING (../../data/hodanje/) ---
Upstairs: 6/13 windows (46.2%)
Walking: 5/13 windows (38.5%)
Downstairs: 2/13 windows (15.4%)

FINAL APP DECISION: Upstairs


## Why the result is unsatisfactory — and the plan that follows

The recording in `data/hodanje/` is **pure, continuous walking on flat ground** (ground-truth known to the author at capture time). The 12-channel CNN baseline from `03-cnn.ipynb` (`cnn_best.tflite`) splits this single activity across three locomotion classes — Upstairs 46.2 %, Walking 38.5 %, Downstairs 15.4 % — and the majority-vote rule therefore returns the wrong label (`Upstairs`). Even ignoring the vote, fewer than 40 % of the windows are classified correctly, an order of magnitude worse than the in-distribution test result. This is a clear train-test domain shift and the reason the project pivots into a dedicated robustness investigation before any Flutter deployment. Several other user-recordings were tested with similarly unsatisfactory results. 

### What likely went wrong

1. **Device-orientation dependence of the 12-channel input.** Six of the twelve channels (`attitude.roll/pitch/yaw`, `gravity.x/y/z`) encode the *absolute* pose of the phone in the device frame. Rotating the phone around the gravity axis while keeping the physical activity unchanged still rewrites every one of these channels. The MotionSense collection protocol presumeably places the iPhone in a fixed front-pocket position with controlled orientation across all 24 subjects (Malekzadeh et al., 2019; dataset README), so the network never had to learn invariance to in-pocket rotation. The wild Android recordings were made with the phone in different in-pocket poses, and the network reads that as a different activity — exactly the failure mode Henpraserttae et al. (2011) flag when arguing for orientation-independent features.
2. **Cross-platform conversion is not a substitute for invariance.** The Android → iOS CoreMotion mapping in Section 3 (per-axis sign flips on accelerations, pitch and yaw; yaw re-zeroed) only aligns the *coordinate convention* of Sensor Logger with the CMDeviceMotion convention the model was trained on. It cannot align the *device pose* in the pocket. A correct conversion still leaves a model whose inputs depend on orientation.
3. **Walking ↔ stairs is the residual confusion cluster.** The CNN baseline in `03-cnn.ipynb` already reported that `walking`, `upstairs` and `downstairs` form its dominant confusion cluster on held-out MotionSense subjects. Pushing the input distribution off the training manifold redistributes mass *inside that cluster*, which is exactly the pattern observed above.

> Malekzadeh, M., Clegg, R. G., Cavallaro, A., & Haddadi, H. (2019). Mobile sensor data anonymization. *Proceedings of the International Conference on Internet of Things Design and Implementation (IoTDI '19)*, 49-58.
>
> Henpraserttae, A., Thiemjarus, S., & Marukatat, S. (2011). Accurate activity recognition using a mobile phone regardless of device orientation and location. *Proceedings of the 8th International Conference on Body Sensor Networks (BSN 2011)*, 41-46.

### What this triggers in following notebooks 

Rather than declaring `cnn_best.tflite` fit for the Flutter app, the next notebooks form a planned robustness sequence in which each step is the smallest change that can isolate one hypothesis from the failure analysis above:

- **`05-features-classical.ipynb`** — control experiment in the spirit of Hammerla et al. (2016): re-fit SVM/RF on an orientation-invariant statistical-feature set (channel magnitudes, gravity-projection of acceleration, pitch only). Goal: confirm the invariant feature set still carries enough information *before* spending effort on a new model.
- **`06-features-cnn.ipynb`** — 6-channel orientation-invariant CNN trained on the same invariant feature set.
- **`07-in-the-wild-robustness.ipynb`** — replaces the single-recording sanity check above with a per-session macro-F1 comparison between the 12-channel CNN and the 6-channel orientation-invariant CNN on labelled Android sessions. This is the proper version of the test in this notebook.
- **`08-features-classical-walking-frame.ipynb`** / **`09-features-cnn-walking-frame.ipynb`** / **`10-in-the-wild-walking-frame.ipynb`** — stronger invariance via projection of the IMU signals into a *walking frame* aligned with gravity and the dominant horizontal motion direction (Mizell, 2003), trained on the classical and CNN sides, then re-tested in the wild.
- **`11-walking-frame-v2.ipynb`** — fixes a residual 180°-about-gravity sign ambiguity in the v1 walking-frame channels by replacing the signed forward/side projections with their magnitudes, yielding a feature set that is fully invariant to any rotation about gravity.
- **`12-arch-comparison.ipynb`** / **`13-hp-sweep.ipynb`** / **`14-final-evaluation.ipynb`** — architecture comparison over the best invariant feature set, hyperparameter sweep on the chosen architecture, and the final held-out evaluation that produces the model actually shipped to the Flutter app.

> Hammerla, N. Y., Halloran, S., & Plötz, T. (2016). Deep, convolutional, and recurrent models for human activity recognition using wearables. *Proceedings of the 25th International Joint Conference on Artificial Intelligence (IJCAI 2016)*, 1533-1540. arXiv:1604.08880.
>
> Mizell, D. (2003). Using gravity to estimate accelerometer orientation. *Proceedings of the 7th IEEE International Symposium on Wearable Computers (ISWC 2003)*, 252-253.